In [ ]:
import geopandas as gpd
import rasterio
from rasterio import features
from shapely.geometry import shape
import os

# Paths
base_path = r"C:\Users\Moritz\Documents\Uni\Masterarbeit\YETI_MA\Masterarbeit\data"
shapefile_names = [
    "01_AOI_Achensee",
    "02_AOI_Kuehtai",
    "03_AOI_Kaunertal"
]
shapefile_paths = [os.path.join(base_path, "shapefiles", name + ".shp") for name in shapefile_names]
raster_mask_path = os.path.join(base_path, "ses_topo_22_forest_mask_domain_epsg32632.tif")
seen_shp_path = os.path.join(base_path, "shapefiles", "Seen_AOIs.shp")
gletscher_shp_path = os.path.join(base_path, "shapefiles", "GI_3", "Gletscher_AOIs.shp")
output_dir = os.path.join(base_path, "shapefiles", "AOIs_masked")
os.makedirs(output_dir, exist_ok=True)

# Load mask shapefiles
seen_gdf = gpd.read_file(seen_shp_path)
gletscher_gdf = gpd.read_file(gletscher_shp_path)

# Reproject everything to the CRS of the AOI shapefiles
target_crs = gdf.crs  # assuming all AOIs use the same CRS

seen_gdf = seen_gdf.to_crs(target_crs)
gletscher_gdf = gletscher_gdf.to_crs(target_crs)

# Load raster mask as polygons (keep areas where mask == 0)
with rasterio.open(raster_mask_path) as src:
    mask_array = src.read(1)
    mask_shapes = [
        shape(geom)
        for geom, val in features.shapes(mask_array, mask=mask_array == 0, transform=src.transform)
        if val == 0
    ]
    raster_mask_gdf = gpd.GeoDataFrame(geometry=mask_shapes, crs=src.crs)

# Apply masks and save
for shp_path, name in zip(shapefile_paths, shapefile_names):
    gdf = gpd.read_file(shp_path)
    # Intersect with raster mask
    gdf = gpd.overlay(gdf, raster_mask_gdf, how='intersection')
    # Remove lakes
    gdf = gpd.overlay(gdf, seen_gdf, how='difference')
    # Remove glaciers
    gdf = gpd.overlay(gdf, gletscher_gdf, how='difference')
    # Save
    out_path = os.path.join(output_dir, f"{name}_masked.shp")
    gdf.to_file(out_path)

C:\Users\Moritz\AppData\Local\Temp\ipykernel_13304\3355345547.py:39: UserWarning: `keep_geom_type=True` in overlay resulted in 58 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  gdf = gpd.overlay(gdf, raster_mask_gdf, how='intersection')
C:\Users\Moritz\AppData\Local\Temp\ipykernel_13304\3355345547.py:41: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: EPSG:32632
Right CRS: EPSG:31254

  gdf = gpd.overlay(gdf, seen_gdf, how='difference')
C:\Users\Moritz\AppData\Local\Temp\ipykernel_13304\3355345547.py:43: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: EPSG:32632
Right CRS: EPSG:31254

  gdf = gpd.overlay(gdf, gletscher_gdf, how='differe